# Forecast Sell-In — Energy Drink 350ml, Flavored Water 500ml, Natural Juice 1L

Pipeline: agregação dos 3 canais (Supermarkets, Traditional, E-commerce) por Produto+Semana,
engenharia de features (Table 4, sem Holiday_Flag), hold-out das últimas 23 semanas,
modelo **LightGBM** com os parâmetros da Table 3, e previsão **recursiva** no hold-out
(Lag_1w, Lag_4w, Rolling_Mean_4w e Rolling_Std_4w das semanas de teste são recalculados
a partir das PREVISÕES do próprio modelo, não dos valores reais).

> Pré-requisito: `pip install lightgbm pandas numpy matplotlib openpyxl scikit-learn`

Ajuste `SRC` no bloco de configuração para o caminho do seu `Business_Case_Data_Set.xlsx`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from lightgbm import LGBMRegressor

pd.set_option('display.max_columns', None)


## 1. Configuração

In [ ]:
SRC = 'Business_Case_Data_Set.xlsx'   # <-- ajuste o caminho no seu ambiente

PRODS = ['Energy Drink 350ml', 'Flavored Water 500ml', 'Natural Juice 1L']
HOLDOUT_WEEKS = 23

FEATURES = ['Lag_1w', 'Lag_4w', 'Rolling_Mean_4w', 'Rolling_Std_4w',
            'Price_per_kg_USD', 'Price_Change_Pct', 'Numeric_Distribution',
            'Weighted_Distribution', 'Advertising_Investment_USD',
            'Promotion_Investment_USD', 'Promo_Flag', 'Month_Sin', 'Month_Cos',
            'Interaction_Price_Promo']
TARGET = 'Sell_In_Tons_Total'

# Parâmetros EXATOS da Table 3 - Model Parameters (LGBM)
MODEL_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    num_leaves=256,
    min_child_samples=5,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.0,
    reg_lambda=0.0,
    min_split_gain=0.0,
    boosting_type='gbdt',
    objective='regression',
    metric='mape',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)


## 2. Carga e agregação dos dados

Soma os 3 canais por Produto+Semana:
- `Sell_In_Tons_Total`: soma simples do volume
- `Price_per_kg_USD`, `Numeric_Distribution`, `Weighted_Distribution`: **média ponderada pelo volume** de cada canal (evita distorcer quando um canal vende muito mais que os outros)
- `Advertising_Investment_USD`, `Promotion_Investment_USD`: soma (investimento total do produto naquela semana)
- `Promo_Flag`: 1 se **pelo menos um** dos 3 canais teve promoção ativa na semana


In [ ]:
def wavg(group, valcol, wcol):
    w = group[wcol]
    return (group[valcol] * w).sum() / w.sum() if w.sum() != 0 else group[valcol].mean()


def load_and_aggregate():
    df1 = pd.read_excel(SRC, sheet_name='Table 1 - External Variables',
                         keep_default_na=False, na_values=[''])
    df2 = pd.read_excel(SRC, sheet_name='Table 2 - Sell In')

    d2 = df2[df2['Product'].isin(PRODS)].copy()
    d1 = df1[df1['Product'].isin(PRODS)].copy()
    d1['Week'], d2['Week'] = pd.to_datetime(d1['Week']), pd.to_datetime(d2['Week'])

    m = d2.merge(d1, on=['Week', 'Product', 'Channel'], how='left')
    m['Promo_Flag_ch'] = (m['Promotion_Type'] != 'None').astype(int)

    recs = []
    for (prod, week), g in m.groupby(['Product', 'Week']):
        recs.append({
            'Product': prod, 'Week': week,
            'Sell_In_Tons_Total': g['Sell_In_Tons'].sum(),
            'Price_per_kg_USD': wavg(g, 'Price_per_kg_USD', 'Sell_In_Tons'),
            'Numeric_Distribution': wavg(g, 'Numeric_Distribution', 'Sell_In_Tons'),
            'Weighted_Distribution': wavg(g, 'Weighted_Distribution', 'Sell_In_Tons'),
            'Advertising_Investment_USD': g['Advertising_Investment_USD'].sum(),
            'Promotion_Investment_USD': g['Promotion_Investment_USD'].sum(),
            'Promo_Flag': 1 if g['Promo_Flag_ch'].max() > 0 else 0,
        })
    return pd.DataFrame(recs).sort_values(['Product', 'Week']).reset_index(drop=True)


agg = load_and_aggregate()
agg.head()


## 3. Engenharia de features (Table 4, sem FE_09 Holiday_Flag)

Importante: `Rolling_Mean_4w` e `Rolling_Std_4w` usam a janela **t-4 até t-1**
(excluem a semana atual) — por isso aplicamos `shift(1)` antes do `rolling(4)`.


In [ ]:
def engineer_features(agg):
    feat_rows = []
    for prod, g in agg.groupby('Product'):
        g = g.sort_values('Week').reset_index(drop=True)
        s = g['Sell_In_Tons_Total']
        g['Lag_1w'] = s.shift(1)
        g['Lag_4w'] = s.shift(4)
        g['Rolling_Mean_4w'] = s.shift(1).rolling(4).mean()   # t-4..t-1
        g['Rolling_Std_4w'] = s.shift(1).rolling(4).std()     # t-4..t-1
        g['Price_Change_Pct'] = g['Price_per_kg_USD'].pct_change()
        g['Month_Sin'] = np.sin(2 * np.pi * g['Week'].dt.month / 12)
        g['Month_Cos'] = np.cos(2 * np.pi * g['Week'].dt.month / 12)
        g['Interaction_Price_Promo'] = g['Price_per_kg_USD'] * g['Promo_Flag']
        feat_rows.append(g)
    return pd.concat(feat_rows, ignore_index=True)


feat = engineer_features(agg)
feat.tail()


## 4. Treino + forecast recursivo no hold-out

Para cada produto:
1. Treina o LightGBM nas primeiras `n - 23` semanas.
2. No hold-out, a cada semana `t`, recalcula `Lag_1w`, `Lag_4w`, `Rolling_Mean_4w`,
   `Rolling_Std_4w` usando a série `working` — que contém valores reais até o fim do
   treino e **previsões do próprio modelo** a partir daí (walk-forward recursivo).
3. As demais variáveis (preço, promoção, investimento, distribuição, sazonalidade)
   são tratadas como conhecidas com antecedência (inputs de planejamento).


In [ ]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))


def train_and_forecast(feat):
    results, plot_data, models = {}, {}, {}

    for prod in PRODS:
        g = feat[feat['Product'] == prod].sort_values('Week').reset_index(drop=True)
        n = len(g)
        train_end = n - HOLDOUT_WEEKS

        train = g.iloc[:train_end].dropna(subset=FEATURES + [TARGET])
        model = LGBMRegressor(**MODEL_PARAMS)
        model.fit(train[FEATURES], train[TARGET])
        models[prod] = model

        working = g[TARGET].tolist()
        preds = []
        for t in range(train_end, n):
            lag1, lag4 = working[t - 1], working[t - 4]
            roll_window = working[t - 4:t]
            roll_mean = np.mean(roll_window)
            roll_std = np.std(roll_window, ddof=1)

            row = g.iloc[t]
            x = pd.DataFrame([{
                'Lag_1w': lag1, 'Lag_4w': lag4,
                'Rolling_Mean_4w': roll_mean, 'Rolling_Std_4w': roll_std,
                'Price_per_kg_USD': row['Price_per_kg_USD'],
                'Price_Change_Pct': row['Price_Change_Pct'],
                'Numeric_Distribution': row['Numeric_Distribution'],
                'Weighted_Distribution': row['Weighted_Distribution'],
                'Advertising_Investment_USD': row['Advertising_Investment_USD'],
                'Promotion_Investment_USD': row['Promotion_Investment_USD'],
                'Promo_Flag': row['Promo_Flag'],
                'Month_Sin': row['Month_Sin'], 'Month_Cos': row['Month_Cos'],
                'Interaction_Price_Promo': row['Interaction_Price_Promo'],
            }])[FEATURES]

            yhat = max(model.predict(x)[0], 0)
            preds.append(yhat)
            working[t] = yhat  # realimenta a própria previsão

        actual = g[TARGET].iloc[train_end:n].values
        weeks = g['Week'].iloc[train_end:n].values
        m_ = mape(actual, preds)
        results[prod] = {'MAPE': m_, 'Accuracy_1_minus_MAPE': 1 - m_,
                          'n_train': len(train), 'n_holdout': len(preds)}
        plot_data[prod] = pd.DataFrame({'Week': weeks, 'Actual': actual, 'Predicted': preds})

    res_df = pd.DataFrame(results).T.astype(float)
    return res_df, plot_data, models


res_df, plot_data, models = train_and_forecast(feat)
res_df


In [ ]:
overall_actual = np.concatenate([plot_data[p]['Actual'].values for p in PRODS])
overall_pred = np.concatenate([plot_data[p]['Predicted'].values for p in PRODS])
overall_mape = mape(overall_actual, overall_pred)
print(f"Overall (pooled) MAPE: {overall_mape:.4f}   Accuracy (1-MAPE): {1-overall_mape:.4f}")


## 5. Gráfico — Real vs Previsto (hold-out)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 12))
for ax, prod in zip(axes, PRODS):
    d = plot_data[prod]
    ax.plot(d['Week'], d['Actual'], marker='o', color='#2E5C8A', label='Real', linewidth=2)
    ax.plot(d['Week'], d['Predicted'], marker='o', color='#D9534F', label='Previsto (recursivo)',
            linewidth=2, linestyle='--')
    a = res_df.loc[prod]
    ax.set_title(f"{prod} — Hold-out (últimas {HOLDOUT_WEEKS} semanas)  |  "
                 f"MAPE: {a['MAPE']*100:.1f}%  |  Accuracy (1-MAPE): {a['Accuracy_1_minus_MAPE']*100:.1f}%",
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Sell-In (Tons)')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    for label in ax.get_xticklabels():
        label.set_rotation(45)
        label.set_ha('right')
plt.tight_layout()
plt.savefig('holdout_forecast.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Por que a acurácia agregada tende a ficar mais alta

Checagem rápida do efeito de agregação/pooling: o CV (desvio padrão / média) da série
somada nos 3 canais é bem menor do que o de cada canal isolado — isso por si só já explica
boa parte de uma acurácia agregada mais alta do que a de um modelo por canal (Table 5).


In [ ]:
df2 = pd.read_excel(SRC, sheet_name='Table 2 - Sell In')
d2 = df2[df2['Product'].isin(PRODS)].copy()
d2['Week'] = pd.to_datetime(d2['Week'])

for prod in PRODS:
    g = d2[d2['Product'] == prod]
    agg_s = g.groupby('Week')['Sell_In_Tons'].sum()
    print(f"{prod}")
    print(f"  Agregado (3 canais): CV = {agg_s.std()/agg_s.mean():.3f}")
    for ch, gc in g.groupby('Channel'):
        s = gc.set_index('Week')['Sell_In_Tons']
        print(f"  {ch:14s}:     CV = {s.std()/s.mean():.3f}")
    print()
